# Automated variable selection using Factor Analysis

Inspired by https://doi.org/10.1080/10095020.2019.1621549


In [ ]:
from glob import glob
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from factor_analyzer import FactorAnalyzer
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.preprocessing import StandardScaler

## Preprocess the data
Remove unnecessary column, devide the data by total population and standardize it.

In [ ]:
def process_file(path, path_total):
    cols_no_division = [
        "Hustota obyvatel na obytnou plochu",
        "Počet obyvatel na dům",
        "Počet obyvatel na byt",
    ]

    # Load total population data
    total = pd.read_csv(path_total, dtype={"nadzsjd": str}, index_col=0)

    # Load main data
    data = gpd.read_parquet(path)

    # Merge data
    data_total = data.join(total)

    # Drop unnecessary columns
    data_relative = data_total.drop(data.columns[:12], axis=1)

    # Convert columns (except 'geometry') to float
    cols_numeric = data_relative.columns.drop("geometry")
    data_relative[cols_numeric] = data_relative[cols_numeric].astype(float)

    # Normalize by total population (except certain columns)
    population = data_relative["Obyvatelstvo celkem"].replace(
        0, np.nan
    )  # avoid division by zero
    cols_to_normalize = [
        col
        for col in cols_numeric
        if col not in cols_no_division + ["Obyvatelstvo celkem"]
    ]
    data_relative[cols_to_normalize] = data_relative[cols_to_normalize].div(
        population, axis=0
    )

    # Drop rows with NaNs after division
    data_relative = data_relative.dropna(subset=cols_to_normalize)

    # Clip extremely large or small values
    data_relative[cols_to_normalize] = data_relative[cols_to_normalize].clip(-1e6, 1e6)
    data_relative[cols_no_division] = data_relative[cols_no_division].clip(-1e6, 1e6)

    # Replace any remaining infinities with NaN and drop them
    data_relative.replace([np.inf, -np.inf], np.nan, inplace=True)
    data_relative.dropna(subset=cols_to_normalize + cols_no_division, inplace=True)

    # Scale columns
    scaler = StandardScaler()
    data_relative[cols_to_normalize] = scaler.fit_transform(
        data_relative[cols_to_normalize]
    )
    data_relative[cols_no_division] = scaler.fit_transform(
        data_relative[cols_no_division]
    )

    return data_relative

In [ ]:
# path = "/data/uscuni-restricted/04_spatial_census/_modifed_merged_census_2021.parquet"
path = "/data/uscuni-restricted/04_spatial_census/_merged_census_2021.parquet"
path_total = "/data/uscuni-restricted/04_spatial_census/total.csv"

In [ ]:
data_relative = process_file(path, path_total)

In [ ]:
X_processed = data_relative.drop(columns=["Obyvatelstvo celkem", "geometry"])
feature_names = list(X_processed.columns)

# Factor analysis


In [ ]:
#  Determine the number of factors using Kaiser's Criterion
fa_initial = FactorAnalyzer()
fa_initial.fit(X_processed)
eigenvalues, _ = fa_initial.get_eigenvalues()
N = np.sum(eigenvalues >= 1)

print(f"Using Kaiser's Criterion: Selected N = {N} factors.")

In [ ]:
# Plot factors and eigenvalues
plt.figure(figsize=(20, 10))
plt.plot(range(1, len(eigenvalues) + 1), eigenvalues, marker="o")

plt.axhline(1, color="r", linestyle="--")

plt.xlabel("Factors")
plt.ylabel("Eigenvalue")
plt.grid()
plt.show()

### Communality Check

In [ ]:
# Initialize and fit Factor Analysis
fa_n = FactorAnalyzer(n_factors=N)
fa_n.fit(X_processed)
# Get the proportion of variance explained by factors
C1 = fa_n.get_communalities()
C2 = np.mean(C1)
# Keep only variables whose communality is at least the mean
communality_mask = C1 >= C2
vars_after_communality = list(X_processed.columns[communality_mask])
print(f"\nKept {len(vars_after_communality)} variables after communality check.")

### Minimum Spanning Tree Correlation Check

In [ ]:
# Select subset of processed variables
corr_matrix = X_processed[vars_after_communality].corr()

# Define correlation threshold
correlation_threshold = 0.75

# Build a distance-like graph: distance = 1 - |correlation|
graph = 1 - np.abs(corr_matrix)

# Keep only edges above the threshold
graph[graph > (1 - correlation_threshold)] = 0

# Convert graph to a sparse matrix and compute MST
mst = minimum_spanning_tree(csr_matrix(graph)).toarray()

# Extract pairs of variables (edges)
edges = np.argwhere(mst > 0)

# Initialize a set to store variables to remove
vars_to_remove = set()

In [ ]:
if edges.size > 0:
    # Compute the "degree" for each variable in MST
    degrees = np.sum(mst > 0, axis=0) + np.sum(mst > 0, axis=1)

    for u, v in edges:
        var1 = vars_after_communality[u]
        var2 = vars_after_communality[v]

        # Remove the variable with fewer connections
        if degrees[u] < degrees[v]:
            vars_to_remove.add(var1)
        else:
            vars_to_remove.add(var2)

    # Keep only variables not flagged for removal
    vars_keep = [v for v in vars_after_communality if v not in vars_to_remove]
    print(f"Removed {len(vars_to_remove)} variables via MST: {list(vars_to_remove)}")
else:
    print("No highly correlated pairs found.")
    vars_keep = vars_after_communality

In [ ]:
final_variables = vars_keep
print(
    f"\nFinal Set of {len(final_variables)} Variables based on {N} components and correlation filtering:"
)

final_variables